# Streaming scoring demo

The streaming demo runs from the CLI, not from this notebook (a notebook kernel and `awaitTermination` do not mix well). This notebook documents the flow and inspects the scored output afterwards.

**Prerequisites:** a trained model (`make train`). Then, from the repository root, start the scoring job in one terminal:

```bash
poetry run transaction-risk score-stream \
  --input-stream data/streaming/incoming \
  --model models/fraud_risk_pipeline \
  --output data/streaming/scored \
  --checkpoint data/streaming/checkpoints/scoring
```

and drop PaySim-style CSV files into `data/streaming/incoming/` from another terminal. Each micro-batch is scored with the same feature pipeline used in batch mode and appended to `data/streaming/scored/`.

In [1]:
from transaction_risk.spark.session import create_spark_session_from_yaml
spark = create_spark_session_from_yaml('../conf/spark.local.yaml')


In [2]:
# Inspect scored streaming output after the job has processed at least one file
from pyspark.sql import functions as F

scored = spark.read.parquet('../data/streaming/scored')
scored.select('step', 'type', 'amount', 'fraud_probability', 'is_alert', 'stream_batch_id').show(10)
scored.groupBy('stream_batch_id').agg(
    F.count(F.lit(1)).alias('transactions'),
    F.sum('is_alert').alias('alerts'),
).orderBy('stream_batch_id').show()
spark.stop()

+----+--------+--------+--------------------+--------+---------------+
|step|    type|  amount|   fraud_probability|is_alert|stream_batch_id|
+----+--------+--------+--------------------+--------+---------------+
|  88| CASH_IN| 7004.14|0.007408362608739338|       0|              0|
| 144|CASH_OUT| 1427.39|0.005751584123041553|       0|              0|
| 168|TRANSFER| 6809.79|0.006489987364913352|       0|              0|
| 103|CASH_OUT| 6130.05|0.005687937641246754|       0|              0|
| 123|CASH_OUT| 1442.05|0.001621318155001...|       0|              0|
| 157| PAYMENT| 2211.61|0.016056721307360466|       0|              0|
| 222|CASH_OUT|12842.55| 0.00570729206661047|       0|              0|
|  58|TRANSFER| 4830.92| 0.02748377312740158|       0|              0|
| 119|CASH_OUT| 4317.09|0.009950792189695634|       0|              0|
| 131|CASH_OUT|  2497.2|0.001124018662826...|       0|              0|
+----+--------+--------+--------------------+--------+---------------+
only s

+---------------+------------+------+
|stream_batch_id|transactions|alerts|
+---------------+------------+------+
|              0|        5000|   100|
+---------------+------------+------+

